## Project: Cleaning Data for Film Rating Predictive Model

**Goal:** Merge sucessfully the Wikipedia Web Scrapping Data with the 'Full TMDB Movies Dataset (1 Million)' Kaggle Dataset

**Dataset Sources**
1. Kaggle Dataset: Full TMDB Movies Dataset 2024 (1M Movies)
    https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies (accessed: 3/29/2026)
2. Wikipedia Web Scrapping file ('results.csv')



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Loading Data & Filtering

In [ ]:
#loading in Wikipedia results data 
wiki_results = pd.read_csv('results.csv')

In [3]:
wiki_results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16634 entries, 0 to 16633
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Unnamed: 0.1  16634 non-null  int64 
 1   Unnamed: 0    16634 non-null  int64 
 2   title         16634 non-null  object
 3   url           16634 non-null  object
 4   year          16634 non-null  int64 
 5   director      16634 non-null  object
dtypes: int64(3), object(3)
memory usage: 779.8+ KB


In [ ]:
#Load in kaggle datset:

# kaggle = pd.read_csv('TMDB_movie_dataset.csv' , parse_dates=['release_date'], low_memory=False)
# kaggle_filter = kaggle[(kaggle['release_date'].dt.year >= 2020) & (kaggle['release_date'].dt.year <= 2025)] 

# kaggle_filter.info()

In [ ]:
#--Saving new filtered kaggle dataset to a new csv file --

# kaggle_filter.to_csv('tmdb_2020to2025.csv', index=False)
# print("Filtered dataset saved to 'tmdb_2020to2025.csv'")

In [ ]:
#--Loading in new csv file with filtered kaggle data --
tmdb = pd.read_csv('tmdb_2020to2025.csv' , parse_dates=['release_date'], low_memory=False)

### Dataset Cleaning & Normalization 

**Key Insights:**
- Keep only the columns most relevant to our model and drop the rest.
- To allow consistent matching across both datasets, we normalize movie titles by lowercasing, removing punctuation, and trimming subtitles. 

In [ ]:

#tmdb dataset :
    #keeping only columns that we find useful :
    #id , title , vote_count , vote_average , release_date ,revenue , runtime, budget, adult , original_language , overview, popularity, genres , keywords, production_companies
    # idk about production_countries or spoken_languages yet, they will be added but we can always remove them later on 
tmdb_c = tmdb[['id', 'title', 'vote_count', 'vote_average', 'release_date', 'revenue', 'runtime', 'budget', 'adult', 'original_language', 'overview', 'popularity', 'genres', 'keywords', 'production_companies', 'production_countries', 'spoken_languages']].copy()
#tmdb_c.info()
#Extracting only the year from release_date
tmdb_c['year'] = tmdb_c['release_date'].dt.year

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 267852 entries, 0 to 267851
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   id                    267852 non-null  int64         
 1   title                 267851 non-null  object        
 2   vote_count            267852 non-null  int64         
 3   vote_average          267852 non-null  float64       
 4   release_date          267852 non-null  datetime64[ns]
 5   revenue               267852 non-null  int64         
 6   runtime               267852 non-null  int64         
 7   budget                267852 non-null  int64         
 8   adult                 267852 non-null  bool          
 9   original_language     267852 non-null  object        
 10  overview              213373 non-null  object        
 11  popularity            267852 non-null  float64       
 12  genres                179819 non-null  object        
 13 

In [ ]:
#normalizing title for matching with wiki results dataset -> better merging
tmdb_c['title_c'] = tmdb_c['title'].str.lower().str.split(':').str[0].str.split('(').str[0].str.replace(r'[^\w\s]', '', regex=True).str.strip()
tmdb_c = tmdb_c.dropna(subset=['title_c', 'year']) #handle missing values in title_c and year columns


In [41]:
#wiki results dataset :
    #keeping only columns that we find useful :
    #title , url, year, director 
wiki_c = wiki_results[['title', 'url', 'year', 'director']].copy()

#normalize it too
wiki_c['title_c'] = wiki_c['title'].str.lower().str.split(':').str[0].str.split('(').str[0].str.replace(r'[^\w\s]', '', regex=True).str.strip()
wiki_c = wiki_c.dropna(subset=['title_c', 'year'])


### Merging Datasets

Attempt an exact merge on normalized title and year.

In [ ]:
#attempt to merge the two datasets on title and year
merged = pd.merge(tmdb_c, wiki_c, on=['title_c', 'year'], how='inner', suffixes=('_tmdb', '_wiki'))
print(len(merged)) 

#find what did not match

matched = merged[['title_c', 'year']] #get the matched title and year pairs

#  find unmatched tmdb entries by doing a left merge and keeping only left_only entries
unmatched_tmdb = tmdb_c.merge(matched, on=['title_c', 'year'], how='left', indicator=True) 
unmatched_tmdb = unmatched_tmdb[unmatched_tmdb['_merge'] == 'left_only'].drop(columns=['_merge'])#keep only unmatched entries

print(f"Unmatched TMDB: {len(unmatched_tmdb)}")

unmatched_tmdb = unmatched_tmdb.copy()


12374
Unmatched TMDB: 255899


### Fuzzy Matching
- Apply fuzzy matching for titles that did not match exactly in our initial merge.
- For unmatched entries, we use the rapidfuzz library for better matching process. 

In [ ]:
#do fuzzy matching

 #installing rapidfuzz library for fuzzy matching (for better matching)
!pip install rapidfuzz


In [30]:
from rapidfuzz import process, fuzz

In [ ]:
#using fuzzy matching 
def fuzzy_match(title, choices, scorer=80): 
    #if title is empty or null, return None to avoid errors in fuzzy matching
    if not title:
        return None 
    #extractOne = finds best match in list of choices, first element returns if mutiple elemnts have same smiliarity
    #scorer = 80 means that we only consider matches with a similarity score of 80 or higher
    #fuzz.token_sort_ratio = compares two strings and returns a similarity score based on the number of matching tokens, ignoring order and duplicates
    match = process.extractOne(title, choices, scorer=fuzz.token_sort_ratio)
    if match and match[1] >= scorer: #find if a match was found and if its score is above scorer=80
        return match[0]  # return the best matching title
    return None


#apply fuzzy matching to unmatched tmdb titles

#for every unmatched tmdb title, we apply fuzzy matching against the wiki titles and store the matched title in a new column 'matched_title'
unmatched_tmdb['matched_title'] = unmatched_tmdb['title_c'].apply(lambda x: fuzzy_match(x, wiki_titles))
unmatched_tmdb = unmatched_tmdb[unmatched_tmdb['matched_title'].notna()] #keep only those that found a match in wiki titles

#merge again on the new matched title column
fuzzy_m = pd.merge(unmatched_tmdb, wiki_c, left_on='matched_title', right_on='title_c', how='inner', suffixes=('_tmdb', '_wiki'))
fuzzy_m = fuzzy_m[abs(fuzzy_m['year_tmdb'] - fuzzy_m['year_wiki']) <= 1] #keep only those that have a year difference of 1 or less

#we keep only one mtahc per tmdb id which is more popular , so if multiple titles mathc, we keep the one that is most popualr 
#we assume that the most popular one is more likely to be correct 
fuzzy_m = fuzzy_m.sort_values(by='vote_count', ascending=False).drop_duplicates(subset=['id']) #sort by vote count to group similar matches together
fuzzy_m = fuzzy_m.drop_duplicates(subset=['title_c_wiki']) #keep only one match per wiki title

print(f"Fuzzy Matched: {len(fuzzy_m)}")

#combine the merges 
final_merged = pd.concat([merged, fuzzy_m], ignore_index=True)
final_merged = final_merged.drop_duplicates(subset=['id']) #drop any duplicates if any tmdb id matched multiple wiki titles, we keep only one match 
print(f"Final Merged: {len(final_merged)}")


Fuzzy Matched: 6667
Final Merged: 18612


### Final Dataset

Saved to 'final_merged.csv' file. 

In [46]:
print("Unique TMDB IDs:", final_merged['id'].nunique())
print("Unique Wiki titles:", final_merged['title_c_wiki'].nunique())

Unique TMDB IDs: 18612
Unique Wiki titles: 6667


In [47]:
final_merged.to_csv('final_merged.csv', index=False)
print("Saved final_merged.csv")

Saved final_merged.csv
